# 06 - Oracle AI Database 26ai: SQL, vectors and models next to your data

Until now the data lived in parquet files and the models in Python. Here you put both **inside the database**:

1. load the two datasets into relational tables and ask questions in SQL,
2. **Select AI**: ask the same questions in English (Autonomous AI Database only),
3. **AI Vector Search**: turn every player-game into a vector and find "games like this one" with `VECTOR_DISTANCE`,
4. **in-database scoring**: import the scikit-learn model as ONNX and call `PREDICTION_PROBABILITY(...)` from SQL.

It runs on an **Autonomous AI Database 26ai** or on the **Oracle AI Database 26ai Free container** (see `docker/README.md`).
Before you start, set `LOLML_DB_USER`, `LOLML_DB_PASSWORD` and `LOLML_DB_DSN` in the shell that starts Jupyter and run
`python scripts/db_setup.py` once (it creates the `LOLML` user and the tables). Passwords never go into the notebook.

In [ ]:
import os, sys, subprocess, time, warnings
from pathlib import Path
ROOT = Path.cwd().parent if (Path.cwd().parent / "lolml").exists() else Path.cwd()
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "scripts"))   # `lolml` helpers and the db_*.py scripts
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from lolml import data, db, features as F, paths
paths.ensure_dirs()
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)

def run_script(*args):
    """Run one of the workshop scripts with this notebook's Python and show its output."""
    r = subprocess.run([sys.executable, str(ROOT / "scripts" / args[0]), *args[1:]], capture_output=True, text=True, cwd=ROOT)
    print(r.stdout[-3000:] + r.stderr[-2000:])
    r.check_returncode()

## 1. Connect

`lolml.db.connect()` uses python-oracledb in *thin* mode: no Oracle Client to install. It reads the three environment variables (plus `LOLML_DB_WALLET_DIR` / `LOLML_DB_WALLET_PASSWORD` for an mTLS wallet).

In [ ]:
conn = db.connect()
print(db.banner(conn))

## 2. Load the data

`scripts/db_load.py` bulk-loads `PLAYER_MATCH` (one row per player per match, 31 columns picked from `performance`) and
`BY_MINUTE_SNAPSHOT` (the by-minute stats) with `executemany`. It does nothing if the tables are already filled.

In [ ]:
def count(table):
    return conn.cursor().execute(f"select count(*) from {table}").fetchone()[0]

if count("PLAYER_MATCH") == 0 or count("BY_MINUTE_SNAPSHOT") == 0:
    run_script("db_load.py")
pd.DataFrame({"rows": {t: count(t) for t in ("PLAYER_MATCH", "BY_MINUTE_SNAPSHOT")}})

## 3. SQL on the same data

Aggregations that took a `groupby` in pandas are one statement here. Table and column **comments** (see `sql/02_tables.sql`)
are stored in the database; Select AI will use them in the next section.

In [ ]:
db.query_df(conn, """
    SELECT champion_name,
           COUNT(*)                                  AS games,
           ROUND(AVG(win), 3)                        AS win_rate,
           ROUND(AVG(kills + assists) / NULLIF(AVG(deaths), 0), 2) AS kda
    FROM   player_match
    GROUP  BY champion_name
    HAVING COUNT(*) >= 500
    ORDER  BY win_rate DESC
    FETCH FIRST 8 ROWS ONLY
""")

In [ ]:
db.query_df(conn, """
    SELECT table_name, column_name, comments
    FROM   user_col_comments
    WHERE  table_name = 'PLAYER_MATCH' AND column_name IN ('WIN', 'TEAM_POSITION', 'VISION_SCORE', 'TIME_CCING_OTHERS')
    ORDER  BY column_name
""")

## 4. Select AI: ask in English (Autonomous AI Database only)

`sql/03b_select_ai_profile.sql` creates an AI profile that points an OCI Generative AI model at these two tables. It needs
`DBMS_CLOUD_AI`, which only exists on Autonomous AI Database, so the next cell checks first and **skips itself** on the Free
container, or when you have not created the profile yet (set `LOLML_GENAI_KEY`, `LOLML_GENAI_REGION` and `LOLML_GENAI_MODEL`, then run `python scripts/db_setup.py 03`; see the script header).

In [ ]:
def select_ai_ready():
    cur = conn.cursor()
    cur.execute("select count(*) from all_objects where object_name = 'DBMS_CLOUD_AI' and object_type = 'PACKAGE'")
    if cur.fetchone()[0] == 0:
        return "DBMS_CLOUD_AI is not installed in this database (Select AI is Autonomous-only)"
    cur.execute("select count(*) from user_cloud_ai_profiles where profile_name = 'LOL_GENAI'")
    if cur.fetchone()[0] == 0:
        return "profile LOL_GENAI does not exist yet: run  python scripts/db_setup.py 03 (see its header)"
    return None

why_not = select_ai_ready()
if why_not:
    print("Skipped:", why_not)
else:
    cur = conn.cursor()
    cur.execute("begin dbms_cloud_ai.set_profile('LOL_GENAI'); end;")
    question = "which champion has the highest win rate among champions with at least 500 games"
    print("SQL the LLM wrote:\n", cur.execute(f"select ai showsql {question}").fetchone()[0])
    display(db.query_df(conn, f"select ai runsql {question}"))

## 5. AI Vector Search: games like this one

Each player-game becomes a point in a 12-dimensional space: kills, deaths, assists, gold, damage dealt and taken, CS, vision, objective
and turret damage, crowd control and heal/shield, each **per minute** and then z-scored. Games with a similar playstyle end up close
together whatever the champion or the length of the game. `scripts/db_vectors.py` stores 95 % of the matches in `PLAYER_GAME_VECTOR`
(a `VECTOR(12, FLOAT32)` column), saves the mean and standard deviation used for the z-score in `VECTOR_NORM`, and builds an HNSW
index (or an IVF index if the database has no vector memory). The other 5 % of the matches are **held out**: games the database has never seen.

In [ ]:
if count("PLAYER_GAME_VECTOR") == 0:
    run_script("db_vectors.py")
index = db.query_df(conn, "select index_name, index_subtype from user_indexes where table_name = 'PLAYER_GAME_VECTOR' and index_type like '%VECTOR%'")
print(count("PLAYER_GAME_VECTOR"), "stored games; vector index:", index.to_dict("records"))

Take a held-out game, embed it **exactly as the stored games were embedded** (same mean and standard deviation, read back from the
database), and let the database find the 5 nearest neighbours.

In [ ]:
perf = data.load("performance")
assert count("PLAYER_MATCH") == len(perf), "PLAYER_MATCH differs from the dataframe: python scripts/db_load.py --truncate"
holdout_rows = np.flatnonzero(db.is_holdout(perf["match_id"]))
pos = int(np.random.default_rng(7).choice(holdout_rows))
game = perf.iloc[[pos]]
g = game.iloc[0]
print(f"query game: {g['championName']} {g['teamPosition']}  KDA {int(g['kills'])}/{int(g['deaths'])}/{int(g['assists'])}  "
      f"{g['duration']:.0f} min  win={int(g['win'])}")

mean, std = db.load_norm(conn)
query_vector = db.embed(game, mean, std)[0]
pd.Series(query_vector, index=[n for n, _ in db.VECTOR_FEATURES], name="z-score").round(2).to_frame().T

In [ ]:
db.games_like(conn, query_vector, k=5).drop(columns="player_match_id")

The SQL behind it is a plain `ORDER BY VECTOR_DISTANCE(...) FETCH APPROX FIRST 5 ROWS ONLY`. The approximate search answers from the vector index; `EXACT` scans every row. Same neighbours, very different cost:

In [ ]:
print(db.GAMES_LIKE_SQL.format(mode="APPROX", k=5).strip())
rows = {}
for mode, exact in (("index (APPROX)", False), ("full scan (EXACT)", True)):
    db.games_like(conn, query_vector, 5, exact)                      # warm-up
    t0 = time.time()
    for _ in range(5):
        found = db.games_like(conn, query_vector, 5, exact)
    rows[mode] = {"ms per query": round((time.time() - t0) / 5 * 1000, 1), "ids": sorted(found.player_match_id)}
pd.DataFrame(rows).T

In [ ]:
sim = db.games_like(conn, query_vector, k=50)
print(f"the 50 most similar games won {sim['win'].mean():.0%} of the time; this game was a {'win' if g['win'] == 1 else 'loss'}")
sim.groupby("team_position").size().sort_values(ascending=False).to_frame("neighbours")

## 6. Score rows with a model *inside* the database (ONNX)

Train in Python, export to ONNX, let the database run it. `scripts/export_onnx.py` trains a gradient boosting classifier on the honest
post-game stats that are columns of `PLAYER_MATCH` (`lolml.features.post_game_numeric`, outcome columns removed) and exports it with
`skl2onnx`. `DBMS_DATA_MINING.IMPORT_ONNX_MODEL` imports it as the model `LOL_WIN_POSTGAME`, and SQL calls
`PREDICTION_PROBABILITY`. The cell skips itself if this database cannot import ONNX models.

In [ ]:
supports_onnx = conn.cursor().execute(
    "select count(*) from all_arguments where package_name = 'DBMS_DATA_MINING' and object_name = 'IMPORT_ONNX_MODEL'").fetchone()[0] > 0
if not supports_onnx:
    print("Skipped: DBMS_DATA_MINING.IMPORT_ONNX_MODEL is not available in this database")
else:
    name = db.ONNX_MODELS["postgame"][0]
    if not (paths.MODELS / f"{name.lower()}.onnx").exists():
        run_script("export_onnx.py")                                  # train + export (about a minute on the full data)
    import db_score                                                    # scripts/db_score.py
    sidecar = db_score.stage_and_import(conn, "postgame")
    print(db.query_df(conn, "select model_name, mining_function, algorithm from user_mining_models").to_string(index=False))

In [ ]:
if supports_onnx:
    display(db.query_df(conn, """
        SELECT champion_name, team_position, team_id,
               kills || '/' || deaths || '/' || assists                      AS kda,
               ROUND(PREDICTION_PROBABILITY(lol_win_postgame, 1 USING *), 3) AS p_win,
               win
        FROM   player_match
        WHERE  match_id = (SELECT MIN(match_id) FROM player_match WHERE MOD(match_id, 20) = 7)
        ORDER  BY team_id, team_position
    """))
    display(db.query_df(conn, """
        SELECT COUNT(*) AS holdout_rows,
               ROUND(AVG(CASE WHEN PREDICTION(lol_win_postgame USING *) = win THEN 1 ELSE 0 END), 4) AS accuracy_in_sql
        FROM   player_match WHERE MOD(match_id, 20) = 7
    """))

Is the database really computing the same numbers as Python? Fetch the hold-out rows, run the very same ONNX file with `onnxruntime`, and compare:

In [ ]:
if supports_onnx:
    db_score.parity(conn, "postgame", sidecar, limit=20000)

## 7. Clean up

Nothing to do if you want to keep exploring. To drop every object this workshop created in the `LOLML` schema, run
`python scripts/db_setup.py 06`. To remove the Free container and its data: `docker compose -f docker/compose.yaml down -v`.

In [ ]:
conn.close()